# Beer Servings Regression Model

This notebook trains a regression model on the beer servings dataset and saves a model file for a Streamlit app. The target column is `total_litres_of_pure_alcohol`.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path('beer-servings (1).csv')
df = pd.read_csv(data_path)
df = df.dropna(subset=['total_litres_of_pure_alcohol']).copy()
df.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
import joblib

features = ['beer_servings', 'spirit_servings', 'wine_servings', 'continent']
target = 'total_litres_of_pure_alcohol'

df = df.dropna(subset=[target]).copy()
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['beer_servings', 'spirit_servings', 'wine_servings']
categorical_features = ['continent']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=300, random_state=42))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print('MAE:', round(mae, 3))
print('R2 Score:', round(r2, 3))

joblib.dump(model, 'beer_alcohol_model.joblib')
print('Model saved as beer_alcohol_model.joblib')

In [ ]:
sample = pd.DataFrame([{
    'beer_servings': 120,
    'spirit_servings': 80,
    'wine_servings': 50,
    'continent': 'Europe'
}])

pred = model.predict(sample)[0]
print(f'Predicted total litres of pure alcohol: {pred:.2f}')